# A huge flow network — solving 1000+ elements

The other examples load hand-built showcase cases (tens of elements). This one
**generates** a much larger network *programmatically* and solves its steady
**mean flow** as a scale stress test of the solver, then visualizes the result and
extracts its end-to-end perturbation behaviour.

Two generators are provided, each a single self-contained routine — pick one with
the `BUILDER` switch below:

| builder | topology |
|---|---|
| **`build_spine`** | a single mass-flow **spine** with modules chained along it: recursive **manifolds** (`splitter → { leaf chains \| nested manifolds } → junction`), **serial** `[loss, duct]` runs, and **sudden-area-change** chains. Grows in *depth*. One inlet, one outlet. |
| **`build_mesh`** | a dense layered **lattice**: each node is one static-pressure **plenum** (`junction`) carrying many inflows and outflows at once, with consecutive layers wired by a wrap-around fan so every plenum mixes several feeds and splits into several branches. Each branch is a resistive `loss` then a `duct`. Grows in *width × depth*; supports **multiple inlets and outlets**. |

Both are acyclic split/join graphs — the regime the homotopy stress tests cover —
just far larger. The solve runs from a quiescent (zero-flow) start with no regime
or flow-direction prescribed.

In [ ]:
import sys, os, time
from collections import Counter, deque, defaultdict
import random

_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "nefes")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

import numpy as np
import plotly.graph_objects as go

from nefes.thermo.configure import perfect_gas
from nefes.elements import catalog as cat
from nefes.elements.ids import RESIDUAL_NAMES
from nefes.shell.network import Network
from nefes.assembly.derive import ES_M, ES_P, ES_T, ES_MDOT, ES_PT
from nefes.plotting import use_nefes_theme, COLORWAY

# Below is because plotly LaTeX rendering is not working out of box in notebooks viewed in VSCode/Cursor.
from IPython.display import display, HTML
import plotly.offline as pyo
pyo.init_notebook_mode()
display(HTML(
    '<script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.7/MathJax.js?config=TeX-AMS-MML_SVG"></script>'
))

use_nefes_theme()   # register + activate the modern Nefes plotly template

# ---------------------------------------------------------------------------
# CONTROL
#   BUILDER          : "spine" (deep, 1 in / 1 out) or "mesh" (dense lattice,
#                      multiple in / out).  Both are single routines defined in
#                      section 1.
#   TARGET_ELEMENTS  : approximate element count.  Each builder appends whole
#                      modules / layers until it crosses this count, so the
#                      realised total lands a little above it.  Try 200 for a
#                      quick run, 10000 to push the mean-flow solve and the map
#                      hard.  (The perturbation cell at the end is meaningful only
#                      at moderate sizes -- see its note.)
BUILDER = "mesh"
TARGET_ELEMENTS = 1000
# ---------------------------------------------------------------------------

# Air, reference scales, and the operating point.
R_AIR, GAMMA = 287.0, 1.4
CP = GAMMA * R_AIR / (GAMMA - 1.0)
P_REF, T_REF = 101325.0, 300.0
MDOT, A_MAIN = 20.0, 0.50

## 1. The generators

Each builder is **one routine** that takes a target element count and returns a
ready-to-solve `Network` — call it and go. They append modules / layers until the
element count crosses `target`, so the realised size lands just above it.

- `build_spine` chains modules along a single feed→dump spine; its nested helpers
  (`serial`, `sac`, `manifold`) re-branch recursively, so the graph grows **deep**.
- `build_mesh` stacks rows of static-pressure plenums wired by a wrap-around fan,
  so the graph grows **wide and densely interconnected**, with as many inlets and
  outlets as asked. Each plenum is a *single* `junction` carrying many inflows and
  outflows at once (the residual equates static pressure across all ports and
  conserves mass over all of them); the resistive `loss` on every branch makes the
  flow split determinate.

In [ ]:
def build_spine(target, seed=7, area=A_MAIN, mdot=MDOT):
    """A deep feed→dump spine of nested manifolds, serial runs, and SAC chains.

    Modules are chained along one mass-flow spine until ``target`` elements exist.
    Nested helpers (closed over ``rng``): ``serial`` appends a constant-area
    ``[loss, duct]`` run; ``sac`` appends a sudden-area-change chain alternating
    area; ``manifold`` lays a ``splitter → { serial leaf | nested manifold } →
    junction`` fan and recurses to ``depth``.  Returns a ready-to-solve Network.
    """
    rng = random.Random(seed)
    net = Network(gas=perfect_gas(R_AIR, GAMMA), p_ref=P_REF, T_ref=T_REF, mdot_ref=mdot)

    def serial(src, a, n_blocks):
        prev = src
        for _ in range(n_blocks):
            lo = net.add(cat.loss(round(rng.uniform(0.2, 1.2), 3), name="loss"))
            net.connect(prev, lo, a)
            du = net.add(cat.duct(length=round(rng.uniform(0.1, 0.8), 3), name="duct"))
            net.connect(lo, du, a)
            prev = du
        return prev

    def sac(src, a, n_sac):
        a_hi, a_lo = a, a * 0.7              # alternate area about `a` (even n_sac)
        prev = src
        for i in range(n_sac):
            s = net.add(cat.sudden_area_change(cc=round(rng.uniform(0.6, 0.95), 3), name="sac"))
            net.connect(prev, s, a_hi if i % 2 == 0 else a_lo)
            prev = s
        return prev                          # closes on a_hi -> caller's edge sits at `a`

    def manifold(feed, a, depth):
        s = net.add(cat.splitter(name="split"))
        net.connect(feed, s, a)
        ends = []
        for _ in range(rng.randint(3, 5)):
            a_b = round(a * rng.uniform(0.6, 0.9), 4)
            if depth > 0 and rng.random() < 0.5:
                ends.append((manifold(s, a_b, depth - 1), a_b))
            else:
                ends.append((serial(s, a_b, rng.randint(2, 5)), a_b))
        j = net.add(cat.junction(name="join"))
        for end, a_b in ends:
            net.connect(end, j, a_b)
        return j

    cur = net.add(cat.mass_flow_inlet(mdot, T_REF, name="feed"))
    plan = ["manifold", "serial", "sac", "manifold", "manifold", "serial"]
    k = 0
    while len(net._elements) < target:
        kind = plan[k % len(plan)]; k += 1
        if kind == "manifold":
            cur = manifold(cur, area, depth=2)
        elif kind == "serial":
            cur = serial(cur, area, rng.randint(4, 8))
        else:
            cur = sac(cur, area, 2 * rng.randint(2, 5))
    net.connect(cur, net.add(cat.pressure_outlet(P_REF, name="dump")), area)
    return net


def build_mesh(target, width=10, n_inlets=2, n_outlets=2, fanout=3, seed=11,
               area=A_MAIN, mdot=MDOT):
    """A dense layered lattice of plenums joined by resistive branches.

    Each lattice node is one static-pressure ``junction`` carrying many inflows and
    many outflows at once; consecutive rows are wired by a wrap-around fan (node
    ``w`` → nodes ``w-1 … w+1`` of the next row, mod ``width``), so every interior
    plenum mixes ``fanout`` feeds and splits into ``fanout`` branches.  Each branch
    is a ``loss`` (pressure drop + a determinate split) followed by a ``duct``
    (length, so the perturbation acoustics carry a real phase per branch).  ``n_inlets`` mass-flow feeds and ``n_outlets`` pressure
    drains attach through fan splitters / junctions.  Rows are stacked until the
    element count crosses ``target``.  Returns a ready-to-solve Network.
    """
    rng = random.Random(seed)
    net = Network(gas=perfect_gas(R_AIR, GAMMA), p_ref=P_REF, T_ref=T_REF, mdot_ref=mdot)

    def branch(a_node, b_node):             # a_node -> loss -> duct -> b_node
        a = round(area * rng.uniform(0.5, 1.0), 4)
        lo = net.add(cat.loss(round(rng.uniform(0.3, 1.5), 3), name="branch"))
        net.connect(a_node, lo, a)
        du = net.add(cat.duct(length=round(rng.uniform(0.2, 1.0), 3), name="line"))
        net.connect(lo, du, a)
        net.connect(du, b_node, a)

    def row():
        return [net.add(cat.junction(name="plenum")) for _ in range(width)]

    rows = [row()]                                          # layer 0 plenums
    for i in range(n_inlets):                               # inlet fan onto a slice
        s = net.add(cat.splitter(name="infan"))
        net.connect(net.add(cat.mass_flow_inlet(mdot / n_inlets, T_REF, name="feed")), s, area)
        for w in range(i, width, n_inlets):
            branch(s, rows[0][w])

    while len(net._elements) < target:                     # interior lattice rows
        prev = rows[-1]; cur = row(); rows.append(cur)
        for w in range(width):
            for d in range(fanout):
                branch(prev[w], cur[(w + d - fanout // 2) % width])

    last = rows[-1]
    for j in range(n_outlets):                             # outlet fan from a slice
        c = net.add(cat.junction(name="outfan"))
        for w in range(j, width, n_outlets):
            branch(last[w], c)
        net.connect(c, net.add(cat.pressure_outlet(P_REF, name="drain")), area)
    return net

## 2. Build it

The element-type census shows the mix the generator produced.

In [ ]:
build = {"spine": build_spine, "mesh": build_mesh}[BUILDER]
net = build(target=TARGET_ELEMENTS)

census = Counter(RESIDUAL_NAMES[e.residual_id] for e in net._elements)
print(f"builder  : {BUILDER}")
print(f"target   : {TARGET_ELEMENTS}")
print(f"elements : {len(net._elements)}")
print(f"edges    : {len(net._edges)}")
print("-" * 34)
for name, n in census.most_common():
    print(f"  {name:<22} {n}")

## 3. Solve the mean flow

`Network.solve()` compiles the immutable problem and runs the damped-Newton solve
through its vanishing-friction homotopy (`stab = 0.1 -> 0.01 -> 0`), warm-started
from a dead stop. `verbose=1` prints one line per homotopy stage.

In [ ]:
t0 = time.perf_counter()
sol = net.solve(verbose=1)
dt = time.perf_counter() - t0

prob = net.compile()
print("-" * 34)
print("converged   :", sol.converged)
print("Newton iters:", sol.iterations)
print("||R_hat||   :", f"{sol.residual_norm:.2e}")
print("unknowns    :", 3 * prob.n_edges, f"  ({prob.n_edges} edges x 3)")
print("wall time   :", f"{dt:.2f} s")

## 4. Is the answer physical?

For a converged steady state we expect: every edge strictly subsonic (the v1
scope), strictly positive pressure and temperature everywhere, and global mass
conservation — the mass-flow feed in must equal the net draw at the pressure
outlet. We read the boundary edges from the solver's own **terminals**.

In [ ]:
from nefes.perturbation import find_terminals

est = sol.table()
M, p, T = np.abs(est[ES_M]), est[ES_P], est[ES_T]

print(f"max |M|   : {M.max():.4f}   (subsonic: {M.max() < 1.0})")
print(f"min p     : {p.min():.1f} Pa   (> 0: {p.min() > 0})")
print(f"min T     : {T.min():.2f} K    (> 0: {T.min() > 0})")
print(f"finite x  : {np.all(np.isfinite(sol.x))}")

terms = find_terminals(prob, sol.x)
inlet_edge  = [t.edge for t in terms if t.at_tail][0]
outlet_edge = [t.edge for t in terms if not t.at_tail][0]
m_in  = sum(sol.edge(t.edge)["mdot"] for t in terms if t.at_tail)
m_out = sum(sol.edge(t.edge)["mdot"] for t in terms if not t.at_tail)
print("-" * 34)
print(f"terminals : {len(terms)}   (inlet edge {inlet_edge}, outlet edge {outlet_edge})")
print(f"feed in   : {m_in:+.6f} kg/s")
print(f"dump out  : {m_out:+.6f} kg/s")
print(f"imbalance : {m_in - m_out:+.2e} kg/s   (machine zero)")

## 5. Field statistics

Two scalar cuts of the converged state. **Edge Mach distribution:** the spine and
the metering branches sit at distinct bands, all well below 1. **Total-pressure
march:** sorting `p_t` high→low traces the dissipative drop from the feed, through
every loss, junction and area change, down to the dump.

In [ ]:
edge_M = np.abs(est[ES_M])
edge_pt = est[ES_PT]

fig = go.Figure()
fig.add_trace(go.Histogram(x=edge_M, nbinsx=40, marker_color=COLORWAY[0]))
fig.update_layout(
    title=f"Edge Mach distribution across {prob.n_edges} edges (all subsonic)",
    xaxis_title="|M|", yaxis_title="edge count", showlegend=False, height=380,
)
fig.show()

order = np.argsort(edge_pt)[::-1]
fig2 = go.Figure()
fig2.add_trace(go.Scatter(y=edge_pt[order] / 1e3, mode="lines",
                          line=dict(color=COLORWAY[1], width=2)))
fig2.update_layout(
    title="Edge total pressure, sorted feed -> dump",
    xaxis_title="edge rank (high p_t -> low)", yaxis_title="p_t  [kPa]", height=380,
)
fig2.show()

## 6. Visualizing the network at scale

A node-link **Sankey** is unreadable past a few hundred elements. The structure
that *does* scale is a **layered DAG layout**: place every element at an *x* given
by its longest-path depth from a source (how many elements upstream of it), and
spread the elements that share a depth evenly in *y*. Flow then reads strictly
left→right — sources at the left, the spine's recursive manifolds (or the lattice's
plenum rows) fanning into columns, the drains at the right — and the picture stays
bounded no matter how many elements there are. Each edge is colored by its Mach
number.

The layout is pure connectivity (one topological pass), so it costs nothing even
at tens of thousands of elements; only the link *lines* are dropped past a few
thousand edges (they turn to clutter), leaving the Mach-colored edge markers.

In [ ]:
def layered_layout(n_nodes, edges):
    """(x, y, depth) per node: x = longest-path depth from a source, y = rank in level."""
    succ = [[] for _ in range(n_nodes)]
    indeg = [0] * n_nodes
    for t, h, _a in edges:
        succ[t].append(h)
        indeg[h] += 1
    depth = [0] * n_nodes
    ind = indeg[:]
    dq = deque(i for i in range(n_nodes) if ind[i] == 0)
    while dq:                                  # Kahn topological sweep -> longest path
        u = dq.popleft()
        for v in succ[u]:
            depth[v] = max(depth[v], depth[u] + 1)
            ind[v] -= 1
            if ind[v] == 0:
                dq.append(v)
    by_level = defaultdict(list)
    for i in range(n_nodes):
        by_level[depth[i]].append(i)
    y = np.zeros(n_nodes)
    for nodes in by_level.values():
        for r, i in enumerate(nodes):
            y[i] = (r + 0.5) / len(nodes)
    depth = np.array(depth)
    return depth / max(depth.max(), 1), y, depth


xpos, ypos, depth = layered_layout(prob.n_nodes, net._edges)
print(f"layered layout: {prob.n_nodes} nodes, longest path = {depth.max()} elements deep")

tail = np.array([t for (t, _h, _a) in net._edges])
head = np.array([h for (_t, h, _a) in net._edges])
mid_x, mid_y = 0.5 * (xpos[tail] + xpos[head]), 0.5 * (ypos[tail] + ypos[head])

fig = go.Figure()
if prob.n_edges <= 4000:   # link lines: informative while sparse, clutter past a few thousand
    lx = np.empty(3 * prob.n_edges); ly = np.empty(3 * prob.n_edges)
    lx[0::3], lx[1::3], lx[2::3] = xpos[tail], xpos[head], np.nan
    ly[0::3], ly[1::3], ly[2::3] = ypos[tail], ypos[head], np.nan
    fig.add_trace(go.Scattergl(x=lx, y=ly, mode="lines",
                               line=dict(width=0.5, color="rgba(140,150,170,0.30)"),
                               hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scattergl(
    x=mid_x, y=mid_y, mode="markers",
    marker=dict(size=4, color=np.abs(est[ES_M]), colorscale="Turbo",
                colorbar=dict(title="|M|"), showscale=True),
    customdata=np.stack([np.arange(prob.n_edges), np.abs(est[ES_MDOT]), np.abs(est[ES_M])], axis=1),
    hovertemplate="edge %{customdata[0]:.0f}<br>|mdot|=%{customdata[1]:.3g} kg/s<br>M=%{customdata[2]:.3f}<extra></extra>",
    showlegend=False,
))
fig.update_layout(
    title=f"Network map — {prob.n_nodes} elements, {prob.n_edges} edges (feed left -> dump right)",
    xaxis=dict(title="longest-path depth (normalized)", showgrid=False),
    yaxis=dict(title="element rank within depth", showgrid=False, zeroline=False),
    height=620,
)
fig.show()

## 7. Perturbation analysis at scale

On the **same** converged mean flow the compiled network supports a linear
**perturbation** analysis (theory §12): three characteristics — two acoustic
(`f`, `g`) and the entropy wave (`h`) — propagate on the mean state. The `duct`
elements carry the phase factor `e^{iωL/c}`; `loss`/`junction` branches act only
algebraically. So the acoustic response is genuinely **frequency-dependent** and
worth sweeping (a duct-free network would give a flat, frequency-independent
response).

We drive every terminal with a unit incoming acoustic wave and read the
whole-network **scattering matrix** `S(ω)` at each frequency — the
boundary-independent descriptor — then report **throughput**: how many
frequencies we sweep per second. The size-normalized rate (**edge·freq/s**) is
roughly constant with network size, so the linear solve scales about linearly
(with mild super-linear fill-in from the sparse factorization).

**Is the perturbation physical?** Two convention-free checks:

- **Bounded & stable.** A dissipative interior closed by passive terminals has no
  undamped resonance, so the worst-case gain `max σ(S)` stays finite (O(1)) across
  the whole band — no blow-up at any real frequency.
- **Causal low-frequency limit.** As `ω → 0` every duct phase `e^{iωL/c} → 1`, so
  the scattering matrix must become **real** (the quasi-steady limit). We check
  `Im S → 0` at a near-DC frequency.

In [ ]:
from nefes.perturbation import perturbation_response, verify_perturbation

# Linear acoustic network on the converged mean flow: one solve per frequency,
# every terminal driven by a unit incoming acoustic wave. Ducts carry the phase,
# so S(omega) is genuinely frequency-dependent.
verify_perturbation(prob, sol.x)                 # subsonic, flow-aligned ducts -> OK
freq = np.linspace(50.0, 250.0, 200)             # Hz

# warm the kernels so the timing reflects steady-state throughput, not compilation
perturbation_response(prob, sol.x, freq[:2], excite=("acoustic",))

t0 = time.perf_counter()
resp = perturbation_response(prob, sol.x, freq, excite=("acoustic",))
dt = time.perf_counter() - t0
rate = freq.size / dt
print(f"swept {freq.size} frequencies over {freq[0]:.0f}-{freq[-1]:.0f} Hz in {dt:.2f} s")
print(f"throughput         : {rate:.0f} freq/s   ({rate * prob.n_edges:,.0f} edge*freq/s)")

# whole-network scattering matrix S(omega) -> worst-case acoustic gain per frequency
S = resp.multiport_scattering_matrix()           # (n_freq, n_out, n_in)
sigma = np.linalg.svd(S, compute_uv=False).max(axis=1)

# physicality (convention-free)
S_dc = perturbation_response(                     # near-DC: duct phases -> 1, S -> real
    prob, sol.x, np.array([1.0e-2]), excite=("acoustic",)
).multiport_scattering_matrix()[0]
print(f"terminals          : {len(terms)}  (scattering matrix {S.shape[1]}x{S.shape[2]})")
print(f"bounded / stable   : worst-case gain max sigma(S) = {sigma.max():.3f} over the band")
print(f"causal omega->0    : max|Im S| = {np.abs(S_dc.imag).max():.2e} at 0.01 Hz "
      f"(vs {np.abs(S[0].imag).max():.2f} at {freq[0]:.0f} Hz)")

use_nefes_theme()
fig = go.Figure()
fig.add_trace(go.Scattergl(x=freq, y=sigma, mode="lines",
                           line=dict(color=COLORWAY[0]), name="max sigma(S)"))
fig.update_layout(
    title=f"Acoustic gain spectrum - {len(terms)}-terminal scattering matrix, {prob.n_edges} edges",
    xaxis_title="frequency [Hz]",
    yaxis_title="worst-case gain  max sigma(S)",
)
fig.show()

## Takeaway

A 1000+ element network — either a deep `spine` of recursive manifolds, serial
loss/duct runs and sudden-area-change chains, or a dense `mesh` lattice of plenums
joined by loss+duct branches with multiple inlets and outlets — built
programmatically from a single routine and solved to a converged,
strictly-subsonic, mass-conserving steady state in a handful of Newton iterations
from a quiescent start, with **no** regime or flow-direction prescribed. The
layered-DAG **network map** and the **flow cascade** scale to tens of thousands of
elements where a Sankey cannot; set `BUILDER` and `TARGET_ELEMENTS` at the top to
dial topology and size. On top of the same converged state, the linear
**perturbation** layer sweeps the acoustic scattering matrix across a frequency
band at hundreds of edge·freq per millisecond, scaling about linearly, and the
response is bounded across the band and reduces to the real quasi-steady limit as
`ω → 0` — physical at scale.

## Export for FNetLibUI

The full network is available as `network` (a `Network`) and its converged mean flow as `sol` (a `Solution`).
Save either to a UI-readable YAML — `sol.to_yaml(path)` embeds the mean-flow result fields, `network.to_yaml(path)` writes the topology only — then open the file in FNetLibUI.

In [ ]:
import os, tempfile

network, sol = sol.network, sol  # the primary Network and its mean-flow Solution
_out = os.path.join(tempfile.mkdtemp(), "huge_network_stress.yaml")
sol.to_yaml(_out)  # embeds the mean-flow results; use network.to_yaml(_out) for topology only
print("saved case:", _out)